# ROCOv2 Bangla: API server for the Android app

This notebook serves your fine-tuned model (`qwen_lora_bangla_radiology`) to the **ROCOv2 Bangla** Android app.

* Sections 1–4 are copied unchanged from `ROCOv2_Bangla_Qwen35_4B_inference_app.ipynb` (install, settings, Google Drive, model, prompt + `explain()` + `side_check()`).
* Sections 5–10 add the API: a FastAPI server in a background thread, a free cloudflared HTTPS tunnel, and a QR code that the app scans.

**How to run:** *Runtime › Change runtime type › T4 GPU*, then run the cells from top to bottom. Section 5 and section 9 are required; 6, 7 and 8 are optional checks. Keep this tab open while you use the app.

**Warning:** student research prototype trained on 231 verified rows (208 train / 23 held out). It can make medical mistakes (wrong side, wrong organ). It is not a diagnosis. The tunnel is public, so do not use real patient images.

### 1. Install
Same install as the training notebook. It uses `%%capture`, so it looks silent for a few minutes. Wait for the finished tick.

In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
# Unsloth bundles the gated delta net kernels; a leftover pip fla would shadow them
!uv pip uninstall -qqq flash-linear-attention fla-core
# causal_conv1d is supported only on torch==2.8.0. If you have newer torch versions, please wait 10 minutes!
!uv pip install --no-build-isolation causal_conv1d==1.6.0
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

### 2. Settings

In [ ]:
ADAPTER_PATH = "/content/drive/MyDrive/qwen_lora_bangla_radiology"   # folder you saved to Google Drive in the training notebook
HF_DATASET   = "eltorio/ROCOv2-radiology"
HF_SPLIT     = "test"     # your IDs look like ROCOv2_2023_test_..., so use "test"

### 3. Connect Google Drive and load the fine-tuned model

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
assert os.path.isdir(ADAPTER_PATH), f"Adapter folder not found: {ADAPTER_PATH}. Check the folder name in Google Drive."
print(os.listdir(ADAPTER_PATH))   # should show adapter_config.json and adapter_model.safetensors

In [ ]:
from unsloth import FastVisionModel
import torch

# Loading the adapter folder also loads the base model Qwen3.5-4B that it was trained on.
model, tokenizer = FastVisionModel.from_pretrained(
    ADAPTER_PATH,
    load_in_4bit = False,                    # same setting as training
)
FastVisionModel.for_inference(model)         # switch to inference mode (faster, no training)
print("Model loaded.")

### 4. Prompt and generate function
The prompt is the same Bangla instruction used with Gemini and used in training. The model was trained **with the English caption in the prompt**, so answers are best when a caption is given. Without a caption it uses the image only, which it was not trained for, so expect weaker answers.

In [ ]:
PROMPT_WITH_CAPTION = """তুমি একজন অভিজ্ঞ ডাক্তার।

এই মেডিকেল ছবি এবং ক্যাপশন দেখে গ্রামের সাধারণ মানুষের জন্য সহজ বাংলায় মাত্র ২-৩টি ছোট বাক্যে লিখো।

ক্যাপশন:
{caption}

নিয়ম:

১। কঠিন মেডিকেল শব্দ লিখবে না।

২। শুধু বলবে—

• শরীরের কোন অংশ

• কী সমস্যা

৩। চিকিৎসা লিখবে না।

৪। শুধুই বাংলা লিখবে।
"""

PROMPT_IMAGE_ONLY = """তুমি একজন অভিজ্ঞ ডাক্তার।

এই মেডিকেল ছবি দেখে গ্রামের সাধারণ মানুষের জন্য সহজ বাংলায় মাত্র ২-৩টি ছোট বাক্যে লিখো।

নিয়ম:

১। কঠিন মেডিকেল শব্দ লিখবে না।

২। শুধু বলবে—

• শরীরের কোন অংশ

• কী সমস্যা

৩। চিকিৎসা লিখবে না।

৪। শুধুই বাংলা লিখবে।
"""

def make_prompt(caption):
    caption = (caption or "").strip()
    if caption:
        return PROMPT_WITH_CAPTION.format(caption = caption)
    return PROMPT_IMAGE_ONLY

DISCLAIMER_BN = "এটি শুধু ছবি বোঝার জন্য সাহায্য। এটি ডাক্তারের পরামর্শ নয়। অনুগ্রহ করে ডাক্তার দেখান।"

def explain(image, caption = ""):
    """image = PIL image, caption = English caption (can be empty). Returns Bangla text."""
    messages = [
        { "role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": make_prompt(caption)} ] }
    ]
    input_text = tokenizer.apply_chat_template(messages, add_generation_prompt = True, enable_thinking = False)
    inputs = tokenizer(
        image.convert("RGB"),
        input_text,
        add_special_tokens = False,
        return_tensors = "pt",
    ).to("cuda")
    out = model.generate(
        **inputs,
        max_new_tokens = 512,
        use_cache = True,
        do_sample = False,          # greedy: same answer every time, no random-sampling crash
        repetition_penalty = 1.05,
    )
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens = True)[0].strip()

In [ ]:
import re

def side_check(caption, bangla):
    """Simple safety check: does left/right in the Bangla answer match the English caption?"""
    if not caption:
        return "No caption given, so left/right could not be checked."
    cap_left  = bool(re.search(r"\bleft\b",  caption, re.I))
    cap_right = bool(re.search(r"\bright\b", caption, re.I))
    out_left  = any(w in bangla for w in ["বাম", "বাঁ"])
    out_right = "ডান" in bangla
    if cap_left and not cap_right and out_right and not out_left:
        return "WARNING: caption says LEFT but the Bangla says RIGHT. Do not trust this answer."
    if cap_right and not cap_left and out_left and not out_right:
        return "WARNING: caption says RIGHT but the Bangla says LEFT. Do not trust this answer."
    return "Left/right check passed (or no side mentioned). This does not check anything else."

## Part B: API server for the Android app
Everything above is the inference notebook unchanged. The cells below add the server. Do **not** run the Gradio cell from the old notebook in this session: two front-ends would share one T4.

### 5. Install the server packages and get the server code
Installs `fastapi`, `uvicorn`, `python-multipart` and `qrcode` with every package that is already installed pinned to its current version, so torch, transformers, trl and unsloth cannot change. Then it downloads the server code from GitHub.

In [ ]:
import re, subprocess, sys, importlib.metadata as md

PINNED_BEFORE = {p: md.version(p) for p in ("torch", "transformers", "trl", "unsloth")}

SERVER_PKGS = ["fastapi>=0.110", "uvicorn>=0.29", "python-multipart>=0.0.9", "qrcode[pil]>=7.4"]
UPGRADABLE = {"fastapi", "starlette", "uvicorn", "python-multipart", "qrcode"}
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
pins = [line for line in freeze.splitlines()
        if re.match(r"^[A-Za-z0-9_.\-]+==", line)
        and line.split("==")[0].lower().replace("_", "-") not in UPGRADABLE]
with open("/content/server_constraints.txt", "w") as f:
    f.write("\n".join(pins) + "\n")
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/server_constraints.txt", *SERVER_PKGS],
                   capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
assert r.returncode == 0, "pip install failed (nothing was upgraded). See the message above."
after = {p: md.version(p) for p in PINNED_BEFORE}
assert after == PINNED_BEFORE, f"Pinned stack changed! before={PINNED_BEFORE} after={after}"
print("OK. Model stack unchanged:", after)

# Server code (FastAPI app, job queue, model wrapper) from the project repository.
REPO = "https://github.com/fryfahim/Cse499A.git"
REPO_REF = "claude/rocov2-bangla-android-app-1mhf6u"   # change to "main" after the pull request is merged
subprocess.run(["rm", "-rf", "/content/rocov2_app"], check=True)
subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", REPO_REF, REPO, "/content/rocov2_app"], check=True)
if "/content/rocov2_app/server" not in sys.path:
    sys.path.insert(0, "/content/rocov2_app/server")
print("Server code ready in /content/rocov2_app/server")

### 6. Check the server code against this notebook and connect the model
The server uses its own byte-for-byte copy of the prompts, `DISCLAIMER_BN` and `side_check()`. This cell stops with an error if they differ from the ones defined above. It then wraps the already-loaded `model` and `tokenizer` (no reload).

Settings:
* `MAX_IMG_SIDE = 512`: the server shrinks images exactly like training did (`convert("RGB")` then `thumbnail((512, 512))`), which is also how the fine-tune notebook evaluated the held-out rows. `None` = full resolution, like the old upload cell.
* `CONTENT_ORDER = IMAGE_FIRST`: the order used by `explain()` above. `TEXT_FIRST` = the order of the training conversations.

In [ ]:
import importlib
import notebook_logic, worker, inference, app as server_app, colab_runtime
for _m in (notebook_logic, worker, inference, server_app, colab_runtime):
    importlib.reload(_m)   # picks up a fresh `git clone` when you re-run cell 5
from inference import make_explain_fn, IMAGE_FIRST, TEXT_FIRST
from app import ServerConfig, create_app

assert notebook_logic.PROMPT_WITH_CAPTION == PROMPT_WITH_CAPTION, "PROMPT_WITH_CAPTION differs"
assert notebook_logic.PROMPT_IMAGE_ONLY == PROMPT_IMAGE_ONLY, "PROMPT_IMAGE_ONLY differs"
assert notebook_logic.DISCLAIMER_BN == DISCLAIMER_BN, "DISCLAIMER_BN differs"
for _cap in ["", "  Large left-sided pneumothorax ", "Focal right upper lobar opacity on chest X-ray"]:
    assert notebook_logic.make_prompt(_cap) == make_prompt(_cap)
    for _bn in ["বাম ফুসফুসে সমস্যা", "ডান ফুসফুসে সমস্যা", "বুকের ছবি"]:
        assert notebook_logic.side_check(_cap, _bn) == side_check(_cap, _bn)
print("Server prompts, disclaimer and side_check() are identical to this notebook.")

# ---- server settings --------------------------------------------------------
MAX_IMG_SIDE  = 512           # None = full resolution
CONTENT_ORDER = IMAGE_FIRST   # or TEXT_FIRST
PORT          = 8000
TUNNEL        = "cloudflared" # or "ngrok" (needs NGROK_AUTHTOKEN in Colab Secrets)
# -----------------------------------------------------------------------------

explain_fn = make_explain_fn(model, tokenizer, content_order=CONTENT_ORDER)
config = ServerConfig(max_img_side=MAX_IMG_SIDE)
print("Model connected. Image shrink:", MAX_IMG_SIDE, "| content order:", CONTENT_ORDER)

### 7. (Optional) Get demo images onto your phone
Downloads a few ROCOv2 test images by ID, plus their English captions, into Google Drive folder **`rocov2_demo_images`**. It reads the dataset one shard (~450 MB) at a time and stops as soon as it has them, so the full dataset (tens of GB) is not downloaded.

On the phone, open the Google Drive app → `rocov2_demo_images` → ⋮ → **Download** for each image. The captions are in `captions.txt`; copy the matching one into the app.

`ROCOv2_2023_test_000384` is a real left/right failure from the fine-tune notebook's held-out results. The caption says RIGHT, the model answered বাম (LEFT), and the app should show the **amber** warning.

In [ ]:
from colab_runtime import DEMO_SAMPLES, fetch_rocov2_samples

DEMO_DIR = "/content/drive/MyDrive/rocov2_demo_images"
found = fetch_rocov2_samples(list(DEMO_SAMPLES), DEMO_DIR)
for image_id, (path, caption) in found.items():
    print(f"{image_id}  ({DEMO_SAMPLES[image_id]})\n   file:    {path}\n   caption: {caption}\n")
missing = set(DEMO_SAMPLES) - set(found)
if missing:
    print("Not found:", missing)

### 8. (Optional, recommended once) Same answer as `explain()`? How long does it take?
Runs the model three times on one image: the notebook's own `explain()`, the server code at full resolution, and the server code with the 512 px shrink. The first two must give **identical** text. The times are real measurements on this GPU; nothing about speed is assumed. In float32 on a T4 this can take a few minutes.

It uses `ROCOv2_2023_test_000384` from section 7 if you ran it, otherwise it asks you to upload an image.

In [ ]:
import io, time
from PIL import Image

if "found" in globals() and "ROCOv2_2023_test_000384" in found:
    _path, PARITY_CAPTION = found["ROCOv2_2023_test_000384"]
    PARITY_BYTES = open(_path, "rb").read()
else:
    from google.colab import files
    _up = files.upload()
    PARITY_BYTES = _up[next(iter(_up))]
    PARITY_CAPTION = input("English caption (optional, press Enter to skip): ")

t0 = time.time()
theirs = explain(Image.open(io.BytesIO(PARITY_BYTES)), PARITY_CAPTION)
t1 = time.time()
ours_full = make_explain_fn(model, tokenizer, content_order=IMAGE_FIRST)(Image.open(io.BytesIO(PARITY_BYTES)), PARITY_CAPTION)
t2 = time.time()
ours_server = explain_fn(server_app.preprocess_image(Image.open(io.BytesIO(PARITY_BYTES)), MAX_IMG_SIDE), PARITY_CAPTION)
t3 = time.time()

print("Server code == notebook explain() (full resolution):", ours_full == theirs)
print(f"Seconds: explain() {t1 - t0:.1f} | server code full-res {t2 - t1:.1f} | server code {MAX_IMG_SIDE}px {t3 - t2:.1f}")
print("\nWhat the app will show:\n", ours_server)
print("\nside_check:", side_check(PARITY_CAPTION, ours_server))
assert ours_full == theirs, "Server code does not match explain(). Do not serve until this is fixed."

### 9. Start the API and show the QR code
Starts the server in the background (the notebook stays usable), opens a free HTTPS tunnel, checks `/health` locally and through the public URL, and prints the **URL**, a new random **API key** and a **QR code**.

In the app: **Settings → Scan QR** (or paste the URL and key) → **Test connection**.

Every run of this cell makes a new URL and key, so scan again after re-running it.

In [ ]:
import subprocess, sys
from colab_runtime import (new_api_key, start_server, local_health, start_cloudflared,
                           start_ngrok, public_health, show_connection)

for _name in ("SERVER", "TUNNEL_PROC"):   # re-running: stop the previous server and tunnel
    try:
        globals()[_name].stop()
    except (KeyError, AttributeError):
        pass

API_KEY = new_api_key()
fastapi_app = create_app(explain_fn, side_check, api_key=API_KEY, config=config)
SERVER = start_server(fastapi_app, port=PORT)
print("local /health:", local_health(PORT))

if TUNNEL == "ngrok":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", "/content/server_constraints.txt", "pyngrok"], check=True)
    PUBLIC_URL = start_ngrok(PORT)
else:
    TUNNEL_PROC = start_cloudflared(PORT)
    PUBLIC_URL = TUNNEL_PROC.url

print("public /health:", public_health(PUBLIC_URL))
show_connection(PUBLIC_URL, API_KEY)
print("\nKeep this tab open while you use the app.")

### 10. (Optional) Send one real job through the public URL
The same request the app makes. Prints the end-to-end time and the server's model time.

In [ ]:
import requests, time

if "PARITY_BYTES" not in globals():
    from google.colab import files
    _up = files.upload(); PARITY_BYTES = _up[next(iter(_up))]
    PARITY_CAPTION = input("English caption (optional): ")

hdr = {"X-API-Key": API_KEY}
t0 = time.time()
r = requests.post(f"{PUBLIC_URL}/api/v1/jobs", headers=hdr, timeout=60,
                  files={"image": ("image.jpg", PARITY_BYTES)}, data={"caption": PARITY_CAPTION})
print(r.status_code, r.json())
job_id = r.json()["job_id"]
while True:
    s = requests.get(f"{PUBLIC_URL}/api/v1/jobs/{job_id}", headers=hdr, timeout=30).json()
    if s["status"] in ("completed", "failed"):
        break
    time.sleep(2)
print(f"end-to-end: {time.time() - t0:.1f} s")
if s["result"]:
    res = s["result"]
    print(f"model time: {res['elapsed_ms'] / 1000:.1f} s | mode: {res['mode']} | side check: {res['side_check']['level']}")
    print(res["bangla_text"])
else:
    print("failed:", s["error"])

### Stop the server

In [ ]:
SERVER.stop()
try:
    TUNNEL_PROC.stop()
except NameError:
    pass
print("Server and tunnel stopped. The app will show Offline.")

### Notes
* **Colab disconnects** after idling and after about 12 hours. Then run everything again from the top. The adapter on Google Drive stays. The URL and key will be new, so scan the new QR code.
* **Privacy:** uploads are held in memory only and dropped after inference. The server does not log images, captions or answers. Traffic still passes through Cloudflare's public tunnel, so do not use real patient images.
* cloudflared's log is in `/content/cloudflared.log`.